In [83]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [84]:
df = pd.read_csv("../data/cleaned_cpu_scheduling_process.csv")
df.head()

,arrival_time,burst_time,priority,process_type,best_algorithm,burst_priority_ratio,interaction_1,interaction_2
0,0,1,5,IO-bound,SJF,0.166667,5,0
1,20,14,9,IO-bound,Round Robin,1.400000,126,280
2,11,15,8,IO-bound,Priority,1.666667,120,165
3,4,18,7,IO-bound,Round Robin,2.250000,126,72
4,20,13,5,CPU-bound,Round Robin,2.166667,65,260


In [85]:
df["relative_burst"] = df["burst_time"] / (df["burst_priority_ratio"] + 1)
df["arrival_burst_ratio"] = df["arrival_time"] / (df["burst_time"] + 1)
df["priority_burst_interaction"] = df["priority"] / (df["burst_time"] + 1)
df["soft_short"] = (df["burst_time"] < 7).astype(int)
df["arrival_stability"] = df["arrival_time"] / (df["burst_time"] + 1)
df["sjf_vs_srtf_signal"] = df["burst_time"] / (df["arrival_time"] + 1)
df['interaction_1'] = df['burst_time'] * df['priority']
df['interaction_2'] = df['arrival_time'] * df['burst_time']
df['burst_priority_ratio'] = df['burst_time'] / (df['priority'] + 1)




In [86]:
features = [
    "arrival_time",
    "burst_time",
    "priority",
    "burst_priority_ratio",
    "interaction_1",
    "interaction_2",
    "process_type",
    "soft_short",
    "arrival_stability",
    "sjf_vs_srtf_signal",
    "relative_burst",
    "arrival_burst_ratio",
    "priority_burst_interaction"

]

X = df[features]

# Encode target
le = LabelEncoder()
y = le.fit_transform(df["best_algorithm"])

In [87]:
numerical_features = [
    "arrival_time",
    "burst_time",
    "priority",
    "burst_priority_ratio",
    "interaction_1",
    "interaction_2",
    "arrival_stability",
    "sjf_vs_srtf_signal",
    "relative_burst",
    "arrival_burst_ratio",
    "priority_burst_interaction"
]

binary_features = ["soft_short"]

categorical_features = ["process_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("bin", "passthrough", binary_features)
    ]
)

In [88]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [89]:
model = XGBClassifier(
    n_estimators=500,
    scale_pos_weight=2,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_lambda=1,
    reg_alpha=0.1,
    random_state=42
)

In [90]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", model)
])

In [91]:
pipeline.fit(X_train, y_train)

c:\Users\SOHAM\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:03:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['arrival_time', 'burst_time',
                                                   'priority',
                                                   'burst_priority_ratio',
                                                   'interaction_1',
                                                   'interaction_2',
                                                   'arrival_stability',
                                                   'sjf_vs_srtf_signal',
                                                   'relative_burst',
                                                   'arrival_burst_ratio',
                                                   'priority_burst_interaction']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['process...
                               feature_types=None, feature_weights=None,
                               gamma=0.1, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [92]:
y_pred = pipeline.predict(X_test)

In [93]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.549

Classification Report:

              precision    recall  f1-score   support

        FCFS       0.52      0.54      0.53       270
    Priority       0.70      0.74      0.72       237
 Round Robin       0.48      0.47      0.48       310
         SJF       0.43      0.29      0.35       129
        SRTF       0.58      0.78      0.67        54

    accuracy                           0.55      1000
   macro avg       0.54      0.57      0.55      1000
weighted avg       0.54      0.55      0.54      1000



In [94]:
df.columns

Index(['arrival_time', 'burst_time', 'priority', 'process_type',
       'best_algorithm', 'burst_priority_ratio', 'interaction_1',
       'interaction_2', 'relative_burst', 'arrival_burst_ratio',
       'priority_burst_interaction', 'soft_short', 'arrival_stability',
       'sjf_vs_srtf_signal'],
      dtype='object')